<a href="https://colab.research.google.com/github/derekhatherley/ds2002-fa26/blob/main/Copy_of_2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [224]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** item in an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [225]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [226]:
# TODO

vendors = pd.DataFrame(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

no_match = joined[joined['_merge'] == 'left_only']
print('Unmatched vendor:', no_match['vendor_id'].unique())
print('Unmatched units:', no_match['qty'].sum())

joined

Unmatched vendor: ['V-07']
Unmatched units: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [227]:
# TODO
joined['name'] = joined['name'].fillna('Unknown Vendor')
q_per_v = joined.groupby('name')['qty'].sum()
q_per_v

# In this table, I made the unmatched vendor show up as unknwon vendor. Even though the vendor ID is known, I elected not to have that
# appear in the table since the rest of the table is filled with real vendor names. If I had not filled the NaN value for the unmatched vendor,
# it would not have appeared in the table at all.

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown Vendor,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [228]:
# TODO
joined['zone'] = joined['zone'].fillna('Unknown Zone')
items_by_zone = joined.groupby('zone')['qty'].sum()
items_by_zone

known_total = items_by_zone.drop('Unknown Zone').sum()
unknown_total = items_by_zone['Unknown Zone']
print('Total from known zones:', known_total)
print('Total from unknown zone:', unknown_total)
print('Difference:', items_by_zone.sum() - known_total)
# Difference of 3 is due to 3 qty being from orders from the unknown vendor which also has an unknown zone
# Total of 4 orders from a known zone, 7 orders if you include the unknown zone

items_by_zone

Total from known zones: 4
Total from unknown zone: 3
Difference: 3


,qty
zone,
A,4
Unknown Zone,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [229]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
add_lines = [{**o, 'lines': o.get('lines') or [{}]} for o in ragged_json]
ragged = pd.json_normalize(add_lines, record_path='lines', meta=['order', 'vendor_id'])
ragged['item'] = ragged['item'].fillna('Unknown Item')
# Leaving NaN qty values so that further calculations can sum the qty
ragged
# Unknown item and fries both have missing quantities. If a quantity of 0 was in the dataframe, it would show up as 0 in the qty column

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Unknown Item,NaN,5,V-10
2,Fries,NaN,6,V-01


### Q6 — Validate

In [230]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item allows for easily interpretable data about each item specifically, like profits and the number of each item purchased. It becomes much easier to groupby and sum values like the total number of each item purchased, or the number of sales by a specific vendor.

b) A missing quantity means that there is no data on the quantity, while a quantity of 0 is a known value that does not require interpretation. Conflating them by, for example, representing them both as 0, means that you are guessing that any unreported or missing data means that there are 0 items, which may or may not be true. This would misrepresent downstream calculations like a sum or a mean.